# Oracle Text and Vector Index Helpers

This notebook uses a real Oracle Database connection only. It demonstrates the two SDK helpers on real Oracle indexes:

- `ensure_oracle_text_index()` creates or verifies an Oracle Text index for native hybrid search.
- `ensure_oracle_vector_index()` creates or verifies an Oracle vector index through `DBMS_VECTOR.CREATE_INDEX`.

The notebook also shows the missing-index preflight error before the helpers run. It creates short-lived real Oracle indexes with unique names, verifies them through `USER_INDEXES`, and drops them at the end unless `ORACLE_KEEP_INDEX_DEMO_INDEXES=1`.

In [ ]:
import os
import uuid
from pathlib import Path

import oracledb

from tavily import TavilyHybridClient
from tavily.databases.oracledb import oracledb as tavily_oracle_db


def load_env_file(path):
    path = Path(path).expanduser().resolve()
    if not path.exists():
        return False
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if value:
            os.environ.setdefault(key, value)
    return True


def load_local_env():
    explicit_path = os.environ.get("ORACLE_DEMO_ENV_FILE")
    if explicit_path and load_env_file(explicit_path):
        print("Loaded environment from ORACLE_DEMO_ENV_FILE")
        return

    cwd = Path.cwd().resolve()
    notebook_dir = Path("examples/oracle").resolve()
    candidates = [
        cwd / ".env",
        cwd / ".venv" / ".env",
        *[parent / ".env" for parent in cwd.parents],
        notebook_dir / ".env",
        *[parent / ".env" for parent in notebook_dir.parents],
    ]

    seen = set()
    for path in candidates:
        path = path.resolve()
        if path in seen:
            continue
        seen.add(path)
        if load_env_file(path):
            print("Loaded environment from", path)
            return

    print("No .env file found; using exported environment variables.")


load_local_env()

required = ["ORACLE_USER", "ORACLE_PASSWORD", "ORACLE_DSN"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise RuntimeError("Missing required Oracle environment variables: " + ", ".join(missing))

TABLE_NAME = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_VECTOR_TABLE", "TAVILY_DOCUMENTS"),
    "ORACLE_VECTOR_TABLE",
)
CONTENT_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_CONTENT_FIELD", "CONTENT"),
    "ORACLE_CONTENT_FIELD",
)
EMBEDDINGS_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_EMBEDDINGS_FIELD", "EMBEDDINGS"),
    "ORACLE_EMBEDDINGS_FIELD",
)
CACHE_TIMESTAMP_FIELD = tavily_oracle_db.validate_identifier(
    os.environ.get("ORACLE_CACHE_TIMESTAMP_FIELD", "ADDED_AT"),
    "ORACLE_CACHE_TIMESTAMP_FIELD",
)
EMBEDDING_DIMENSION = int(os.environ.get("ORACLE_VECTOR_DIMENSION", "1024"))

run_id = os.environ.get("ORACLE_INDEX_DEMO_RUN_ID", uuid.uuid4().hex[:8]).upper()
TEXT_INDEX_NAME = tavily_oracle_db.validate_identifier(f"TVLY_TXT_{run_id}", "TEXT_INDEX_NAME")
VECTOR_INDEX_NAME = tavily_oracle_db.validate_identifier(f"TVLY_VEC_{run_id}", "VECTOR_INDEX_NAME")

connection = oracledb.connect(
    user=os.environ["ORACLE_USER"],
    password=os.environ["ORACLE_PASSWORD"],
    dsn=os.environ["ORACLE_DSN"],
)

print("Connected to Oracle.")
print("Table:", TABLE_NAME)
print("Content column:", CONTENT_FIELD)
print("Vector column:", EMBEDDINGS_FIELD)
print("Text index name:", TEXT_INDEX_NAME)
print("Vector index name:", VECTOR_INDEX_NAME)

In [ ]:
def table_exists(table_name):
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM USER_TABLES WHERE TABLE_NAME = :table_name",
            table_name=table_name,
        )
        return cursor.fetchone()[0] > 0


def table_columns(table_name):
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COLUMN_NAME, DATA_TYPE
            FROM USER_TAB_COLUMNS
            WHERE TABLE_NAME = :table_name
            """,
            table_name=table_name,
        )
        return {row[0]: str(row[1]).upper() for row in cursor.fetchall()}


def ensure_column(column_name, ddl):
    if column_name in table_columns(TABLE_NAME):
        return False
    with connection.cursor() as cursor:
        cursor.execute(f"ALTER TABLE {TABLE_NAME} ADD ({ddl})")
    connection.commit()
    return True


def ensure_real_table():
    if not table_exists(TABLE_NAME):
        with connection.cursor() as cursor:
            cursor.execute(
                f"""
                CREATE TABLE {TABLE_NAME} (
                    ID NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
                    {CONTENT_FIELD} CLOB,
                    {EMBEDDINGS_FIELD} VECTOR({EMBEDDING_DIMENSION}, FLOAT32),
                    {CACHE_TIMESTAMP_FIELD} TIMESTAMP DEFAULT SYSTIMESTAMP
                )
                """
            )
        connection.commit()
        print("Created real Oracle table:", TABLE_NAME)
        return

    added = []
    if ensure_column(CONTENT_FIELD, f"{CONTENT_FIELD} CLOB"):
        added.append(CONTENT_FIELD)
    if ensure_column(EMBEDDINGS_FIELD, f"{EMBEDDINGS_FIELD} VECTOR({EMBEDDING_DIMENSION}, FLOAT32)"):
        added.append(EMBEDDINGS_FIELD)
    if ensure_column(CACHE_TIMESTAMP_FIELD, f"{CACHE_TIMESTAMP_FIELD} TIMESTAMP DEFAULT SYSTIMESTAMP"):
        added.append(CACHE_TIMESTAMP_FIELD)

    print("Table exists:", TABLE_NAME)
    print("Added missing columns:", ", ".join(added) if added else "none")


def print_required_column_types():
    columns = table_columns(TABLE_NAME)
    for column_name in (CONTENT_FIELD, EMBEDDINGS_FIELD, CACHE_TIMESTAMP_FIELD):
        print(column_name, "=>", columns.get(column_name))


ensure_real_table()
print_required_column_types()

## Missing Index Preflight

Before calling the helpers, the notebook checks `USER_INDEXES` for the real index names. Because these names are unique for this run, the first preflight should show the actionable missing-index errors.

In [ ]:
def index_exists(index_name):
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM USER_INDEXES WHERE INDEX_NAME = :index_name",
            index_name=index_name,
        )
        return cursor.fetchone()[0] > 0


def require_index_present(index_name, index_kind, helper_name):
    if not index_exists(index_name):
        raise RuntimeError(
            f"Missing {index_kind} index {index_name}. "
            f"Call {helper_name} before running indexed Oracle retrieval."
        )
    return True


for index_name, index_kind, helper_name in (
    (TEXT_INDEX_NAME, "Oracle Text", "client.ensure_oracle_text_index()"),
    (VECTOR_INDEX_NAME, "Oracle vector", "client.ensure_oracle_vector_index()"),
):
    try:
        require_index_present(index_name, index_kind, helper_name)
        print(f"{index_kind} preflight OK: {index_name}")
    except RuntimeError as exc:
        print(f"{index_kind} preflight error: {exc}")

## Create Real Oracle Indexes

Now call the actual SDK helpers. The Text helper issues real Oracle Text DDL, and the vector helper calls real `DBMS_VECTOR.CREATE_INDEX`.

In [ ]:
def demo_embedding_function(texts, input_type):
    return [[0.0] * EMBEDDING_DIMENSION for _ in texts]


def demo_ranking_function(query, documents, top_n):
    return documents[:top_n]


client = TavilyHybridClient(
    api_key=os.environ.get("TAVILY_API_KEY"),
    db_provider="oracle",
    connection=connection,
    table_name=TABLE_NAME,
    content_field=CONTENT_FIELD,
    embeddings_field=EMBEDDINGS_FIELD,
    text_index_name=TEXT_INDEX_NAME,
    vector_index_name=VECTOR_INDEX_NAME,
    vector_index_type=os.environ.get("ORACLE_VECTOR_INDEX_TYPE", "HNSW"),
    vector_index_distance=os.environ.get("ORACLE_VECTOR_INDEX_DISTANCE", "COSINE"),
    vector_index_neighbors=int(os.environ.get("ORACLE_VECTOR_INDEX_NEIGHBORS", "32")),
    vector_index_efconstruction=int(os.environ.get("ORACLE_VECTOR_INDEX_EFCONSTRUCTION", "200")),
    embedding_function=demo_embedding_function,
    ranking_function=demo_ranking_function,
)

text_created = client.ensure_oracle_text_index()
vector_created = client.ensure_oracle_vector_index()

print("Text index created:", text_created)
print("Vector index created:", vector_created)
print("Text index exists after helper:", index_exists(TEXT_INDEX_NAME))
print("Vector index exists after helper:", index_exists(VECTOR_INDEX_NAME))

## Verify Real Oracle Index Metadata

This cell reads `USER_INDEXES` and `USER_IND_COLUMNS` so the output shows the real Oracle indexes and indexed columns.

In [ ]:
with connection.cursor() as cursor:
    cursor.execute(
        """
        SELECT i.INDEX_NAME,
               i.INDEX_TYPE,
               i.STATUS,
               c.COLUMN_NAME
        FROM USER_INDEXES i
        LEFT JOIN USER_IND_COLUMNS c
          ON i.INDEX_NAME = c.INDEX_NAME
        WHERE i.INDEX_NAME IN (:text_index_name, :vector_index_name)
        ORDER BY i.INDEX_NAME, c.COLUMN_POSITION
        """,
        text_index_name=TEXT_INDEX_NAME,
        vector_index_name=VECTOR_INDEX_NAME,
    )
    rows = cursor.fetchall()

for index_name, index_type, status, column_name in rows:
    print(
        f"index={index_name} type={index_type} status={status} column={column_name}"
    )

## Cleanup

The notebook uses unique real index names so it can reliably show the missing-index path. By default it drops those short-lived indexes after proving they exist. Set `ORACLE_KEEP_INDEX_DEMO_INDEXES=1` to keep them.

In [ ]:
def drop_index_if_exists(index_name):
    if not index_exists(index_name):
        print("Index already absent:", index_name)
        return False
    with connection.cursor() as cursor:
        cursor.execute(f"DROP INDEX {index_name}")
    connection.commit()
    print("Dropped index:", index_name)
    return True


if os.environ.get("ORACLE_KEEP_INDEX_DEMO_INDEXES") == "1":
    print("Keeping demo indexes because ORACLE_KEEP_INDEX_DEMO_INDEXES=1")
else:
    drop_index_if_exists(TEXT_INDEX_NAME)
    drop_index_if_exists(VECTOR_INDEX_NAME)

connection.close()
print("Closed Oracle connection.")